# Short notes on executing Linux, GROMACS, and Python within Jupyter notebook

- Python is a natural environment of Jupyter. Type Python commands as they are.
- To enforce GROMACS and most Linux commands, you should add `!` in front of `gmx` or any other Linux command.
- There are few exceptions, *e.g.*, `cd`. You should use `%` instead of `!`.

In [ ]:
# general imports 
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video

# Why GROMACS?

- Extremely high performance compared to all other molecular dynamics engines.
- User-friendly, all necessary files are written in clear text format.
- No programming involved to use it straight away.
- Huge community of users, forums, tutorials, excellent documentation.
- Both input files and trajectories are independent of hardware used to generate them.
- Compact way of storing trajectory data (.xtc format) - can be read by other tools and libraries for visualization and advanced analyses.
- Large selection of inbox tools for system building (including building proteins from Protein Data Bank) and routine analyses.
- High parallelization capabilities and excellent GPU-based acceleration.
- GROMACS is free software available under the GNU Lesser General Public License.

## GROMACS tutorials

- https://tutorials.gromacs.org/ (official ones)
- http://www.mdtutorials.com/gmx/ (unofficial ones but those were done probably by every person working in GROMACS)

## How to use?

Every GROMACS command starts with <code>gmx</code> prefix. For instance, today we will use following commands: <code>gmx solvate</code>, <code>gmx grompp</code>, <code>gmx mdrun</code>, and <code>gmx energy</code>.

Are there are many <code>gmx</code> commands? Yes! Let's have a look at the full list:

In [ ]:
# !gmx help commands

### Also available here 

- https://manual.gromacs.org/current/user-guide/cmdline.html

## Description of GROMACS commands

If you are interested in details of one particular command, just type <code>gmx help {name_of_the_command}</code> or <code>gmx {name_of_the_command} -h</code>

In [ ]:
!gmx help solvate
# !gmx solvate -h

## Hierarchy of GROMACS files

- `.gro` &mdash; contains atomic coordinates, velocities (optionally), and box dimensions.

- `.top` &mdash; provides the molecular topology information which describes the structure of molecules, the type of atoms, bonds, angles, dihedrals, and non-bonded parameters. It contains the definitions of atom types, bond types, and force field parameters. It can include other topology fragments (often saved with the `.itp` extension) to allow for modularity.

- `.itp` &mdash; individual topology fragments that can be included in the main `.top` file.
 
- `.mdp` &mdash; specifies the parameters for running a simulation. We will discuss `.mdp` parameters throughout the course.

- `.tpr` &mdash; a compiled binary file containing all the information from the files above needed to start a simulation.

# Today's Practical

We will prepare a box with water. This is an example system, which will introduce you to GROMACS, its basic commands and usage, and principal steps of performing molecular simulations.

This exersise is also designed to detail concepts like simulation time, equilibration, and basic analysis necessary to ensure the realibilty of your simulations.

## What files do we have?

In [ ]:
!ls -l

## Short folders' decsription

### force_field

Contains force field ($V(r)$ in the lecture) files necessary to execute our simulations of water/methanol mixtures.

### images

Example images.

### scripts

Contains `.mdp` files we will use today.

### solvents

Starting atomic configurations for water.

### topologies

`.itp` topology of water.

### videos

Example videos.

### week04.ipynb

Jupyter notebook we are using right now.

## Create a folder where we will be working today

In [ ]:
!mkdir playground/

In [ ]:
%cd playground/

In [ ]:
!ls -l

## Water

In [ ]:
!head ../solvents/water_four_site.gro
print('...')
!tail ../solvents/water_four_site.gro

### Four-site model of water

- One additional virtual (ghost) atom on top of an oxygen and two hydrogens
    - necessary to better reproduce hydrogen bonding and dipole moment
    - the virtual atom also represents the lone pairs on the oxygen atom
- Common for water models in order to better reproduce experimental properties
- In biosimulations, four-site water models are used less than three-site (historical reasons mainly)

![water](../images/water_four_site_noMW.png)
![water](../images/water_four_site.png)

### Procedure

- Box of water with 1000 molecules.

### Using `gmx solvate` to prepare initial boxes

- `gmx solvate` command is perfectly suited for our task
- `-cs` &mdash; with that we will fill the box? First, we will prepare boxes with water.
- `-box` &mdash; dimensions/size of the box
- `-maxsol` &mdash; control number of solvent molecules; if more molecules fit into the box than we need
- `-o` &mdash; name of the file we will generate

In [ ]:
!gmx solvate -cs ../solvents/water_four_site.gro -box 3.2 3.2 3.2 -maxsol 1000 -o w1000.gro

In [ ]:
!ls -l

### Renaming files for clarity

In [ ]:
!mv w1000.gro pure_water.gro
!ls -l

<img src='../images/water_box.png' width=600 height=450 alt='water_box'>

### Topology `.top` file

In [ ]:
topology = """; force field parameters
#include "../force_field/forcefield.itp"
#include "../topologies/tip4p_2005.itp"

[ system ]
; name of the system
box of water

[ molecules ]
; compound   #mols
SOL          1000
"""

with open('pure_water.top', 'w') as outFile:
    outFile.write(topology)

In [ ]:
!cat ../topologies/tip4p_2005.itp

In [ ]:
#!cat ../force_field/forcefield.itp

In [ ]:
!cat ../force_field/ffnonbonded.itp

## Running simulations

- Two key commands &mdash; `gmx grompp` and `gmx mdrun`

### `gmx grompp`

- `-f` &mdash; your `.mdp` file that contains a simulation protocol
- `-p` &mdash; your `.top` file
- `-c` &mdash; your `.gro` file
- `-o` &mdash; resulting binary `.tpr` file
- `-maxwarn` &mdash; sometimes GROMACS gives warning that does not allow you to generate a `.tpr`. You can specify max number of warnings allowed. Be careful.
- `-po` &mdash; output text file with used simulation parameters; similar to the input `-f` `.mdp` file but also includes default, not explicitly mentioned options

### `gmx mdrun`

- `-deffnm` &mdash; output files will be called the same as input `.tpr`
- `-v` &mdash; show a live progress of your simulations (we will not use it in some cases today to not overload the Jupyter notebook)
- `-quiet` &mdash; suppress most of the verbose output when executing a command (keep in mind: sometimes better not to suppress them)

## Energy minimization

- Required to ensure that the built system has no steric overlaps/clashes and inappropriate geometries.
- The idea of the algorithm we will use (steepest descent) is to move atoms a bit until there are no large forces.

### `.mdp` options

- https://manual.gromacs.org/current/user-guide/mdp-options.html (full list and description)

In [ ]:
!cat ../scripts/min.mdp

In [ ]:
!gmx grompp -f ../scripts/min.mdp -p pure_water.top -c pure_water.gro -o min_pure_water.tpr -po min_mdout_pure_water.mdp

In [ ]:
!gmx mdrun -deffnm min_pure_water -v

### Minimization/equilibration can generate stepXXX.pdb files 

- Generated to investigate possible issues with the system if any.
- Quite common particularly for minimization.
- Can be deleted if you do not have any troubles at the end.

In [ ]:
!rm -f *step*

### Overwriting the files

- If GROMACS wants to write a file called `w1000.gro` and such already exists, it renames the existing one to `#w1000.gro.1#` before writing a new one.
- Useful to back track the data.
- Can be deleted if you are sure you do not need them.

In [ ]:
!rm -f *#*

In [ ]:
!ls -l

## Hierarchy of output simulation files

- `.gro` &mdash; contains **final** atomic coordinates, velocities (optionally), and box dimensions.

- `.xtc` &mdash; a compressed trajectory file containing the atomic coordinates at different time frames throughout the simulation. Due to its compression, it is smaller in size compared to other trajectory formats, but it does not store velocities or forces. **99% choice for biosimulations.**

- `.trr` &mdash; a full precision trajectory file. Unlike the `.xtc` file, it contains atomic coordinates, velocities, and forces without compression. This makes it larger in size, but it can be more useful for certain analysis tasks where precision is crucial. **Almost never used in biosimulations.**

- `.edr` &mdash; stores various energy terms (e.g., kinetic energy, potential energy, total energy, temperature, pressure) and other system information throughout the simulation.
 
- `.log` &mdash; a text file that provides a detailed record of the simulation progress, including settings, energy terms at each step, and any warnings or errors.

- `.cpt` &mdash; a checkpoint file that contains all necessary information to continue a simulation from where it left off.

## Analysing simulations

- Multiple built-in GROMACS tools are available
- There are some that used almost always

### `gmx energy`

-  Tool to work with a binary `.edr` energy file
- `-f` &mdash; input `.edr` file
- `-o` &mdash; output `.xvg` file; you can then plot it &mdash we will do it now.

In [ ]:
!echo 'Potential' | gmx energy -f min_pure_water.edr -o pot_min_pure_water.xvg

In [ ]:
!head -n 50 pot_min_pure_water.xvg

In [ ]:
potential = np.loadtxt('pot_min_pure_water.xvg', comments=['#', '@'])
plt.plot(potential[:, 0], potential[:, 1], 'r*')
plt.legend(['minimization'])
plt.xlabel('Iteration')
plt.ylabel('Potential Energy  (kJ/mol)')
plt.show()

### More about matplotlib

https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html

## Equilibration

- **Every** system first needs to be equilibrated. The more complex systems, the longer simulations times are required.
- For our system (pure water), we do not need long equilibration, just short one to avoid any steric problems if any are unresolved after the energy minimzation.

### Equilibration with constant volume (NVT ensemble)

In [ ]:
!cat ../scripts/eq_nvt.mdp

### Do we save every time step in a simulation trajectory?

- It is not necessary in the majority of simulations, especially biological ones.
- You can, of course, still do it, but then you generate terabytes of data.
- Usually, trajectories are saved every 1&ndash;100 ps depending on a system.

### Simulation with every time step being stored

In [ ]:
Video('../videos/video_every_time_step.mp4', width=800, height=600, embed=True)

In [ ]:
!gmx grompp -f ../scripts/eq_nvt.mdp -maxwarn 1 -p pure_water.top -c min_pure_water.gro -o eqnvt_pure_water.tpr -po eqnvt_mdout_pure_water.mdp

In [ ]:
# !gmx mdrun -deffnm eqnvt_pure_water -v
!bash ../../wolf/run_md_wolf32.sh eqnvt_pure_water

In [ ]:
!echo 'Temperature' | gmx energy -f eqnvt_pure_water.edr -o temp_eqnvt_pure_water.xvg

In [ ]:
!head -n 50 temp_eqnvt_pure_water.xvg

In [ ]:
temperature = np.loadtxt('temp_eqnvt_pure_water.xvg', comments=['#', '@'])
plt.plot(temperature[:, 0], temperature[:, 1], 'r*')
plt.axhline(y=298.15, color='b', linestyle='--')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Temperature (K)')
plt.show()

## TASK

Calculate the potential energy from `eqnvt` simulation and plot it.

### Equilibration with constant pressure (NPT ensemble)

In [ ]:
!cat ../scripts/eq_npt.mdp

In [ ]:
!gmx grompp -f ../scripts/eq_npt.mdp -maxwarn 2 -p pure_water.top -c eqnvt_pure_water.gro -o eqnpt_pure_water.tpr -po eqnpt_mdout_pure_water.mdp

In [ ]:
# !gmx mdrun -deffnm eqnpt_pure_water -v
!bash ../../wolf/run_md_wolf32.sh eqnpt_pure_water

In [ ]:
!echo 'Pressure' | gmx energy -f eqnpt_pure_water.edr -o pres_eqnpt_pure_water.xvg

In [ ]:
pressure = np.loadtxt('pres_eqnpt_pure_water.xvg', comments=['#', '@'])
plt.plot(pressure[:, 0], pressure[:, 1], 'r*')
plt.axhline(y=1, color='b', linestyle='--')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Pressure (bar)')
plt.show()

## TASK

Calculate the potential energy and temperature from `eqnpt` simulation and plot it.

## Production run (in biosimulations, almost always NPT ensemble)

<div class="alert alert-block alert-warning">
Production simulations are where the main results coming from &mdash; this is what you publish
</div>

In [ ]:
!cat ../scripts/prod_md.mdp

In [ ]:
!gmx grompp -f ../scripts/prod_md.mdp -maxwarn 1 -p pure_water.top -c eqnpt_pure_water.gro -o prod_pure_water.tpr -po prod_mdout_pure_water.mdp

## Running long simulations

- We performed a short simulation on a small system.
- Long simulations of a big system usually performed using computers equipped with GPU, computer clusters, supercomputers, etc.
- Advantage: you start a simulation, work on something else, come back later, and the finished simulation is waiting for you.

In [ ]:
# !bash ../../wolf/run_md_wolf32.sh prod_pure_water